### Wi-Fi Sensing

#### Initial Setup

In [29]:
# pandas for data manipulation
# re for regular expressions
import re

# pyplot for plotting
import matplotlib.pyplot as plt

# numpy for numerical operations
import numpy as np
import pandas as pd

# seaborn for advanced plotting
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, make_scorer

# sklearn for machine learning
from sklearn.model_selection import GridSearchCV, train_test_split

# other .py files
from utils.csv_import import get_csv_files_generalistic, sort_meta_info

In [30]:
# Retrieve CSV files

path: str = ("C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\new_data\\16.fev")

FileMap = dict[str, dict[str, str]]
csi_map = dict[str, dict[str, np.ndarray]]

# data_files["user_1"]["a09"]["esp_1"]
# data_files["user_0"]["z00"]["esp_1"][0]
data_files = get_csv_files_generalistic(path)
users_id, posicoes, esp_ids, repetition_ids = sort_meta_info(path)

print(data_files)

{'user_0': {'esp_1': {'w01': WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/16.fev/00-w01-01-01.csv'), 'z00': WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/16.fev/00-z00-01-01.csv')}, 'esp_3': {'w01': WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/16.fev/00-w01-03-01.csv'), 'z00': WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/16.fev/00-z00-03-01.csv')}, 'esp_4': {'w01': WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/16.fev/00-w01-04-01.csv'), 'z00': WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/16.fev/00-z00-04-01.csv')}}}


#### *Def functions*

##### Print functions info

In [31]:
def info(file: FileMap) -> None:
    # Print information about the files dictionaries
    print("Número de ficheiros recolhidos: ", len(file))

    # avoid StopIteration if dict is empty
    esp_count = len(next(iter(file.values()))) if file else 0
    print("ESPs por cenário:", esp_count)

    # Print all scenarios names
    print("\nTodos os cenários:", sorted(file.keys()))

    # Count scenarios by letter prefix
    scenario_prefixes: dict[str, int] = {}
    for scenario in file:
        prefix = scenario[0]
        scenario_prefixes[prefix] = scenario_prefixes.get(prefix, 0) + 1

    print("\nN.º cenários/posição:")
    for prefix, count in sorted(scenario_prefixes.items()):
        print(f"Posição {prefix}: {count} cenários")

    print("---\n")

##### Cálculo módulo (magnitude)

In [32]:
def process_csi(file: str) -> np.ndarray:
    # file read
    csv = pd.read_csv(file, header=None)

    # no da diana são coletadas 120 amostras
    csi_raw: pd.Series = csv.iloc[:, 26]

    total_sc: int = 128
    valid_csi: list[list[float]] = []

    # contar CSI inválidos
    no_match_count: int = 0
    no_complete_count: int = 0

    for entry in csi_raw:
        match = re.search(r"\[(.*?)\]", str(entry))
        if not match:
            no_match_count += 1
            continue
        nums = [float(n) for n in re.findall(r"-?\d+", match.group(1))]
        if len(nums) == total_sc:
            valid_csi.append(nums)
        else:
            no_complete_count += 1

    valid_csi = np.array(valid_csi)

    # (n_amostras, 128)
    complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]

    # coloca sc DC no centro (index 32)
    fft_csi = np.fft.fftshift(complex_csi, axes=1)

    # Extract the 52 active subcarriers (IEEE 802.11n standard): [6:58]
    # (n_amostras, 52)
    active_sc = fft_csi[:, 6:58]

    # Remove subcarriers at positions 25, 26, 27 (center)
    # (n_amostras, 49)
    active_sc = np.delete(active_sc, [25, 26, 27], axis=1)

    # seleciona sub_carriers: 2 a 47
    active_sc = active_sc[:, 2:48]

    # return magnitudes
    return np.abs(active_sc)


# cria o dict magnitudes
# itera sobre cada posicao da grelha
# # para cada posicao, itera sobre cada esp_id e path
# # devolve o dict magnitudes preenchido
def process_magnitudes(files: FileMap) -> dict[str, dict[str, np.ndarray]]:
    magnitudes: dict[str, dict[str, dict[str, np.ndarray]]] = {}

    for user_key, esps_map in files.items():
        if user_key == "user_0":
            continue

        magnitudes[user_key] = {}

        for esp, posicao_map in esps_map.items():
            magnitudes[user_key][esp] = {}

            for posicao, file in posicao_map.items():
                if file is not None:
                    magnitudes[user_key][esp][posicao] = process_csi(file)

    return magnitudes

##### Cálculo da média

In [33]:
# função que calcula a média para cada subportadora (coluna a coluna)
# # para cada esp
def calc_features_per_subcarriers(
    magnitudes: csi_map,
) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray]]:
    media: dict[int, np.ndarray] = {}
    max_value: dict[int, np.ndarray] = {}

    for esp_id in esp_ids:
        esp_key = f"esp_{esp_id}"
        dados_vazios = []

        for user_id in users_id:
            user_key = f"user_{user_id}"

            #if user_key == "user_0":
            #    continue

            # posição z00 de cada user
            dados_z00 = magnitudes[user_key][esp_key].get("z00")
            if dados_z00 is not None:
                dados_vazios.append(dados_z00)

        dados_vazios = np.concatenate(dados_vazios, axis=0)

        # calcular features
        media[esp_key] = np.mean(dados_vazios, axis=0)

        # caso haja valores negativos, utilizamos abs
        max_value[esp_key] = np.max(np.abs(dados_vazios), axis=0)

    return media, max_value

In [34]:
# função retirada do pipeline da Diana
def segment_signal(
    signal: np.ndarray, window_size: int, overlap: int,
) -> list[np.ndarray]:
    segments: list[np.ndarray] = []

    for i in range(0, signal.shape[0], overlap):
        if i + window_size < signal.shape[0]:
            segment = signal[i : i + window_size]
            segments.append(segment)

    return segments


def process_pipeline(
    csi_data: np.ndarray, mean: np.ndarray, max_value: np.ndarray
) -> np.ndarray:
    csi_data_normalized = (csi_data - mean) / max_value

    # parâmetros de segmentação
    # definidos pela Diana
    window_size: int = 10
    overlap_size: int = 2

    segmented_data = segment_signal(csi_data_normalized, window_size, overlap_size)
    segmented_data = np.array(segmented_data)

    # obtemos 2 matrizes de features: mean e std
    features = [np.mean(segmented_data, axis=1), np.std(segmented_data, axis=1)]
    features = np.array(features)

    # rearranja features para o formato (n_segments, 2, 46)
    df_x = np.transpose(features, (1, 0, 2))

    # transforma para (n_segments, 2*46)
    df_x = df_x.reshape(df_x.shape[0], -1)

    return df_x


def define_df(
    esps_map: csi_map, mean: dict[int, np.ndarray], max_value: dict[int, np.ndarray],
) -> np.ndarray:
    user_features = []

    # esps_map = {'esp_1': {'a00': csi_array, 'a01': csi_array, ...}, 'esp_2': {...},}
    for esp_key, posicoes in esps_map.items():
        for posicao, csi in posicoes.items():
            if csi is None:
                continue

            # PARA JÁ - saltamos estas posições
            # podemos alterar para detetar sala vazia (incluir z00)
            if posicao in {"z00", "w00"}:
                continue

            # calcula features para cada csi (posicao da esp_key)
            x = process_pipeline(csi, mean[esp_key], max_value[esp_key])
            user_features.append(x)

    # concatenar apenas junta as matrizes, sem []
    user_features = np.concatenate(user_features, axis=0)

    return user_features

#### *Pre Processing*

Módulo

In [35]:
# info(data_files)

magnitudes: csi_map = process_magnitudes(data_files)

In [37]:
magnitudes

{}

Média para normalizar

In [36]:
# normalização com todos os dados z00
# recolhemos a média/máx para cada esp
# abrangendo os 2 utilizadores

# dict{esp_id: np.ndarray} de size 46
mean_ref: dict[int, np.ndarray] = {}
max_ref: dict[int, np.ndarray] = {}

mean_ref, max_ref = calc_features_per_subcarriers(magnitudes)

KeyError: 'user_0'

In [ ]:
# df format

# user 1
# ------------ ESP 1 --------------------- ESP 2 ------....------------------ TARGET
# ---MEAN[0]---STD[0]---MEAN[1]---STD[1]---....-----MEAN---STD---...................
# --SC1--SC2--SC3....----------------------...----SC1--SC2--SC3....-----------------
# DADOS-------------------------------------------------------------------------1---
# ...
# user 2
# DADOS-------------------------------------------------------------------------2---

df_x = []
df_y = []

for user_key, esps_map in magnitudes.items():
    if user_key == "user_0":
        continue

    # X_user vem com os dados de 1 user
    X_user = define_df(esps_map, mean_ref, max_ref)

    # este append apenas adiciona "1 matriz" de seguida
    df_x.append(X_user)

    # target value
    user_id = int(user_key.split("_")[1])
    n_samples = X_user.shape[0]

    # preencher a coluna target com o id do user
    Y_user = np.full((n_samples,), user_id)
    df_y.append(Y_user)

X_final = np.concatenate(df_x, axis=0)
y_final = np.concatenate(df_y, axis=0)

#### *Machine Learning*

In [ ]:
df = pd.DataFrame(X_final)
df["target"] = y_final

print("df.shape = ", df.shape)
print(df["target"].value_counts())

Divisão treino/teste

In [ ]:
# features
X: np.ndarray = df.drop(columns="target").to_numpy()
y = df["target"].to_numpy()

print("X.shape =", X.shape)
print("y.shape =", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=200,
)

print("X_train.shape =", X_train.shape)
print("X_test.shape  =", X_test.shape)
print("y_train.shape =", y_train.shape)
print("y_test.shape  =", y_test.shape)


*Random Forest Classifier*

In [ ]:
model = RandomForestClassifier(random_state=200)
model = model.fit(X_train, y_train)

pred_test = model.predict(X_test)
accuracy = balanced_accuracy_score(y_test, pred_test)
print("Balanced Accuracy Test Set:", accuracy * 100, "%")

Confusion Matrix

In [ ]:
confusion_matrix_test = confusion_matrix(y_test, pred_test)
sns.set_theme(font_scale=1.2)
plt.figure(figsize=(6, 5))

sns.heatmap(
    confusion_matrix_test,
    annot=True,
    fmt="d",
    cmap="Reds",
    xticklabels=["User 1", "User 2"],
    yticklabels=["User 1", "User 2"],
)

plt.title("Matriz de Confusão")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()
plt.show()


##### Boosting Algorithms

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier


df = pd.DataFrame(X_final)
df["target"] = y_final
X: np.ndarray = df.drop(columns="target").to_numpy()
y = df["target"].to_numpy()

y -= 1

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=200,
)

models = {
    "RF": RandomForestClassifier(n_estimators=300),
    "GB": GradientBoostingClassifier(),
    "XGB": XGBClassifier(eval_metric="mlogloss"),
    "LGBM": LGBMClassifier(),
    "AdaBoost": AdaBoostClassifier(),
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = balanced_accuracy_score(y_test, preds)
    predictions[name] = preds

results


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, preds) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Reds",
        xticklabels=["User 1", "User 2"],
        yticklabels=["User 1", "User 2"],
        ax=axes[idx],
    )
    axes[idx].set_title(f"{name}\nAccuracy: {results[name]:.2%}")
    axes[idx].set_xlabel("Previsto")
    axes[idx].set_ylabel("Real")

axes[-1].axis('off')

plt.tight_layout()
plt.show()


##### Grid Search

In [ ]:
# Grid Search
# 432 combinações
param_grid = {
    "min_samples_split": [4, 16, 32],
    "min_samples_leaf": [4, 16, 32, 64],
    "max_leaf_nodes": [4, 16, 32, 64],
    "n_estimators": [100, 150, 200],
    "max_depth": [4, 16, 32],
}

rf = RandomForestClassifier(
    random_state=200,
    class_weight="balanced_subsample",
    criterion="entropy",
)

scorer = make_scorer(balanced_accuracy_score)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,
    n_jobs=-1,
    verbose=2,
)

# fit
grid_search.fit(X_train, y_train)

# Best model and parameters
print("Best balanced accuracy:", grid_search.best_score_)
print("Best params:", grid_search.best_params_)

# Predict with best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_pred))